## Построение и оценка бейзлайна

Цель данного этапа — оценить, насколько хорошо простые модели способны решать задачу классификации вин, а также задать базовый уровень качества, с которым будут сравниваться более сложные подходы.

В первую очередь строится константный бейзлайн (DummyClassifier), который не использует признаки и всегда предсказывает наиболее частый класс. Это позволяет определить минимальный уровень качества, который должна превосходить любая обучаемая модель.

Далее обучаются несколько базовых моделей из разных семейств:
- логистическая регрессия (линейная модель),
- дерево решений (нелинейная модель),
- метод ближайших соседей (локальная модель).

Для оценки качества используется метрика accuracy, так как классы в датасете распределены относительно равномерно. Это делает accuracy корректной и интерпретируемой метрикой для сравнения моделей.

Все эксперименты проводятся с фиксированным random_state для обеспечения воспроизводимости результатов.

In [1]:
from sklearn.datasets import load_wine
import pandas as pd

data = load_wine()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (142, 13)
Test shape: (36, 13)


Разбиение выполнено с использованием стратификации, что позволяет сохранить распределение классов.
Параметр random_state зафиксирован для воспроизводимости результатов.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

y_dummy = dummy.predict(X_test)
dummy_acc = accuracy_score(y_test, y_dummy)

print("Dummy accuracy:", dummy_acc)

Dummy accuracy: 0.3888888888888889


В качестве бейзлайна используется DummyClassifier, который всегда предсказывает наиболее частый класс.
Это задаёт минимальный уровень качества.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Стандартизация применяется для моделей, чувствительных к масштабу признаков (логистическая регрессия, KNN).



В данном датасете все признаки являются числовыми, поэтому дополнительное кодирование категориальных переменных (например, Label Encoding или One-Hot Encoding) не требуется. Это упрощает этап предобработки и позволяет напрямую применять модели.

In [5]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

y_pred_logreg = logreg.predict(X_test_scaled)
logreg_acc = accuracy_score(y_test, y_pred_logreg)

print("LogReg accuracy:", logreg_acc)

LogReg accuracy: 0.9722222222222222


In [6]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)
tree_acc = accuracy_score(y_test, y_pred_tree)

print("Tree accuracy:", tree_acc)

Tree accuracy: 0.9444444444444444


Дерево решений не требует масштабирования и позволяет учитывать нелинейные зависимости.

In [7]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
knn_acc = accuracy_score(y_test, y_pred_knn)

print("KNN accuracy:", knn_acc)

KNN accuracy: 0.9722222222222222


Метод ближайших соседей чувствителен к масштабу признаков, поэтому применяется после стандартизации.

In [8]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Dummy", "LogReg", "Tree", "KNN"],
    "Accuracy": [dummy_acc, logreg_acc, tree_acc, knn_acc]
})

results.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy
1,LogReg,0.972222
3,KNN,0.972222
2,Tree,0.944444
0,Dummy,0.388889


In [9]:
from sklearn.metrics import classification_report

print("LogReg:\n", classification_report(y_test, y_pred_logreg))
print("Tree:\n", classification_report(y_test, y_pred_tree))
print("KNN:\n", classification_report(y_test, y_pred_knn))

LogReg:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      1.00      0.97        14
           2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

Tree:
               precision    recall  f1-score   support

           0       1.00      0.92      0.96        12
           1       0.88      1.00      0.93        14
           2       1.00      0.90      0.95        10

    accuracy                           0.94        36
   macro avg       0.96      0.94      0.95        36
weighted avg       0.95      0.94      0.94        36

KNN:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      0.93      0.96        14
           2       0.91      1.00      0.95        10

Результаты эксперимента показывают, что константный бейзлайн (DummyClassifier) демонстрирует наименьшее качество, что ожидаемо, поскольку модель не использует информацию о признаках и отражает лишь распределение классов.

Логистическая регрессия показывает высокое качество классификации, что свидетельствует о хорошей линейной разделимости классов в исходном пространстве признаков. Это означает, что основные различия между типами вина могут быть описаны линейными комбинациями химических характеристик.

Модель KNN демонстрирует сопоставимые результаты, что указывает на наличие локальной структуры данных: объекты одного класса действительно расположены ближе друг к другу в признаковом пространстве. Это подтверждает, что схожие по химическому составу вина относятся к одному типу.

Дерево решений также достигает высокого качества, что говорит о наличии нелинейных зависимостей между признаками. Однако такие модели могут быть более склонны к переобучению, особенно на небольших выборках.

В целом, все обучаемые модели существенно превосходят константный бейзлайн, что подтверждает информативность признаков. Наилучший баланс качества и интерпретируемости демонстрирует логистическая регрессия, тогда как KNN и дерево решений позволяют дополнительно учитывать нелинейные и локальные закономерности.

Для обеспечения воспроизводимости эксперимента зафиксирован параметр random_state, а все этапы обработки и обучения выполняются последовательно в рамках одного ноутбука.